In [49]:
import numpy as np
from glob import glob
import os
import skimage as sk
from skan import draw
from skan.csr import skeleton_to_csgraph
from skan import Skeleton, summarize
import napari
import matplotlib
matplotlib.rcParams["image.interpolation"] = 'none'
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import pandas as pd

## Create Skeletons!

In [29]:
cropped_imgs_50_files = sorted(glob(r"Image_Data\Oct_2025_300mm_exp\50_mM\Masked_cropped_images\*.tif"))
cropped_imgs_300_1_files = sorted(glob(r"Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Masked_cropped_images\*.tif"))
cropped_imgs_300_6_files = sorted(glob(r"Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Masked_cropped_images\*.tif"))

cropped_imgs_50 = list(map(sk.io.imread,cropped_imgs_50_files))
cropped_imgs_300_1 = list(map(sk.io.imread,cropped_imgs_300_1_files))
cropped_imgs_300_6 = list(map(sk.io.imread,cropped_imgs_300_6_files))

In [30]:
equalized_hist_50 = [sk.exposure.equalize_adapthist(img) for img in cropped_imgs_50]
equalized_hist_300_1 = [sk.exposure.equalize_adapthist(img) for img in cropped_imgs_300_1]
equalized_hist_300_6 = [sk.exposure.equalize_adapthist(img) for img in cropped_imgs_300_6]

c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 6622 fits in uint16
  return _convert(image, np.uint16, force_copy)
c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 7367 fits in uint16
  return _convert(image, np.uint16, force_copy)
c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 5835 fits in uint16
  return _convert(image, np.uint16, force_copy)
c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting uint32 to uint16 without scaling because max value 8923 fits in uint16
  return _convert(image, np.uint16, force_copy)
c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\util\dtype.py:527: UserWarning: Downcasting

In [31]:
#create binary images using Yen's method
threshold_yen_50 =[img > sk.filters.threshold_yen(img) for img in equalized_hist_50]
threshold_yen_300_1 =[img > sk.filters.threshold_yen(img) for img in equalized_hist_300_1]
threshold_yen_300_6 =[img > sk.filters.threshold_yen(img) for img in equalized_hist_300_6]

c:\ProgramData\anaconda3\envs\skan\lib\site-packages\skimage\filters\thresholding.py:466: RuntimeWarning: divide by zero encountered in log
  crit = np.log(((P1_sq[:-1] * P2_sq[1:]) ** -1) * (P1[:-1] * (1.0 - P1[:-1])) ** 2)


In [9]:
viewer = napari.view_image(cropped_imgs_50[20])
viewer.add_labels(threshold_yen_50[20])

E:\Temp\ipykernel_29796\31700049.py:1: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(cropped_imgs_50[20])


<Labels layer 'Labels' at 0x1f34c562e90>

In [32]:
#remove objects smaller than 30 pixels from binary images
filtered_objects_50 = [sk.morphology.remove_small_objects(img,min_size=50) for img in threshold_yen_50]
filtered_objects_300_1 = [sk.morphology.remove_small_objects(img,min_size=50) for img in threshold_yen_300_1]
filtered_objects_300_6 = [sk.morphology.remove_small_objects(img,min_size=50) for img in threshold_yen_300_6]

In [33]:
fill_small_holes_50 = [sk.morphology.remove_small_holes(img,area_threshold=50) for img in filtered_objects_50]
fill_small_holes_300_1 = [sk.morphology.remove_small_holes(img,area_threshold=50) for img in filtered_objects_300_1]
fill_small_holes_300_6 = [sk.morphology.remove_small_holes(img,area_threshold=50) for img in filtered_objects_300_6]

In [23]:
viewer = napari.view_image(cropped_imgs_300_1[7])
viewer.add_labels(threshold_yen_300_1[7], name="original")
viewer.add_labels(fill_small_holes_300_1[7], name='filled holes')

E:\Temp\ipykernel_29796\5499678.py:1: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(cropped_imgs_300_1[7])


<Labels layer 'filled holes' at 0x1f38969f160>

In [34]:
#create labeled images from filtered binary images
labeled_objects_50 = [sk.morphology.label(img) for img in fill_small_holes_50]
labeled_objects_300_1 = [sk.morphology.label(img) for img in fill_small_holes_300_1]
labeled_objects_300_6 = [sk.morphology.label(img) for img in fill_small_holes_300_6]

In [35]:
#skeletonize labeled images
skeletons_50 = [sk.morphology.skeletonize(img) for img in labeled_objects_50]
skeletons_300_1 = [sk.morphology.skeletonize(img) for img in labeled_objects_300_1]
skeletons_300_6 = [sk.morphology.skeletonize(img) for img in labeled_objects_300_6]

In [37]:
#save skeleton images
names_50 = list(map(os.path.basename,cropped_imgs_50_files))
names_300_1 = list(map(os.path.basename,cropped_imgs_300_1_files))
names_300_6 = list(map(os.path.basename,cropped_imgs_300_6_files))
names = [names_50,names_300_1,names_300_6]

path_50 = r"Image_Data\Oct_2025_300mm_exp\50_mM\Skeletons"
path_300_1 = r"Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Skeletons"
path_300_6 = r"Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Skeletons"
paths = [path_50,path_300_1,path_300_6]

skeletons = [skeletons_50,skeletons_300_1,skeletons_300_6]

for name_set,path,skeleton_set in zip(names,paths,skeletons):
    for name, skeleton in zip(name_set,skeleton_set):
        ubyte_convert = sk.util.img_as_ubyte(skeleton)
        sk.io.imsave(os.path.join(path,name[:-19]+'_skeleton.tif'),ubyte_convert,check_contrast=False)


In [ ]:
#save overlay images of skeletons on equalized hist images
save_path = 'skeletons'
for i in range(len(skeletons)):
    name = os.path.basename(files[i])
    fig,ax=plt.subplots()
    draw.overlay_skeleton_2d(equalized_hist[i],skeletons[i],image_cmap='Greys_r',dilate=1,axes=ax)
    fig.set_tight_layout(tight=True)
    plt.show(block=True)
    fig.savefig(os.path.join(save_path,name[:-4]+'_skeleton_overlay.png'),dpi=300)


### Analyze Skeletons of mito networks

In [38]:
spacing_um = 0.04
branch_data_50 = [summarize(Skeleton(skeleton,spacing=spacing_um),separator='_') for skeleton in skeletons_50]
branch_data_300_1 = [summarize(Skeleton(skeleton,spacing=spacing_um),separator='_') for skeleton in skeletons_300_1]
branch_data_300_6 = [summarize(Skeleton(skeleton,spacing=spacing_um),separator='_') for skeleton in skeletons_300_6]

In [ ]:
#assess dataframe
branch_data_300_1[9].tail()

In [ ]:
#quick visual of all the branch distances organized by branch type
branch_data_300_1[9].hist(column='branch_distance', by='branch_type', bins=100)

In [ ]:
#Quick visual of how the skeletons are labeled by branch type
draw.overlay_euclidean_skeleton_2d(equalized_hist[0], branch_data[0],
                                   skeleton_color_source='branch_type')

In [43]:
#add filename column to each dataframe and merge all dataframes into one
dataframes = [branch_data_50, branch_data_300_1, branch_data_300_6]
save_paths = [r'Image_Data\Oct_2025_300mm_exp\50_mM\Measurements',r'Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Measurements',r'Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Measurements']
conditions = ["50mM","300mM_1.5hr","300mM_6hr"]
for name_set, df_set, path, condition in zip(names,dataframes,save_paths,conditions):
    dfs = []
    for name, df in zip(name_set, df_set):
        df['image_name'] = name[:-19]
        df['condition'] = condition
        dfs.append(df)
    merged_df = pd.concat(dfs,ignore_index=True)
    merged_df.to_csv(os.path.join(path,'skeleton_analysis_'+condition+'.csv'))

In [44]:
na_50_measurements = pd.read_csv(r"Image_Data\Oct_2025_300mm_exp\50_mM\Measurements\skeleton_analysis_50mM.csv")
na_300_1_measurements = pd.read_csv(r"Image_Data\Oct_2025_300mm_exp\300_mM_1_hr\Measurements\skeleton_analysis_300mM_1.5hr.csv")
na_300_6_measurements = pd.read_csv(r"Image_Data\Oct_2025_300mm_exp\300_mM_6_hr\Measurements\skeleton_analysis_300mM_6hr.csv")

In [45]:
all_data = pd.concat([na_50_measurements,na_300_1_measurements,na_300_6_measurements],ignore_index=True)

In [47]:
all_data.to_csv(r'Output\Oct_2025_300mM_exp\all_conditions_skeleton_analysis.csv')

In [46]:
import seaborn as sns

In [ ]:
#plot branch distances for junction to junction branches (branch_type 1) per condition; code from SKAN docs
#added a 'Condition' column based on filename
j2j=(all_data[all_data['branch_type']== 1].
     rename(columns={'branch_distance':
                     'branch distance (um)'}))
per_image = j2j.groupby('image_name').median()
per_image['Condition'] = ['50 mM' if '500mm' in fn else '50 mM' for fn in per_image.index]
sns.stripplot(data=per_image,
              x='Condition',y='branch distance (um)',
              order=['50 mM','500 mM'],
              jitter=True)

TypeError: agg function failed [how->median,dtype->object]

In [ ]:
per_image.to_csv("Output/skeleton_df.csv")